# AetherOS Distributed AI Compute Node (Kaggle T4×2 Edition)

Welcome to the **AetherOS Contributor Compute Worker** (`worker-runtime/aether_worker.py`).

This notebook turns your Kaggle GPU session (dual NVIDIA Tesla T4 GPUs with 32 GB total VRAM) into an authenticated, outbound-polling compute node for the AetherOS platform — executing local PyTorch / HuggingFace `transformers` models (such as **Qwen 2.5 Coder 14B** and **DeepSeek R1 Distill 14B**) without opening inbound ports or calling external AI inference APIs.

---

### Setup Requirements (Kaggle Right Sidebar Settings):
1. **Accelerator**: Select **`GPU T4 x2`**
2. **Internet**: Toggle **`Internet On`** (Required for outbound HTTPS polling to Central AetherOS)
3. **Optional Kaggle Secrets (`Add-ons -> Secrets`)**:
   - `AETHER_REGISTRATION_TOKEN`: Required only if the gateway enforces an enrollment token on `POST /api/workers/register`
   - `AETHER_WORKER_ID` & `AETHER_WORKER_KEY`: Optional pre-registered worker credentials if generated via the AetherOS UI (never paste raw keys directly into printed output)
   - `AETHER_PLATFORM_URL`: Override if connecting to a custom Cloudflare Worker deployment
4. Click **`Run All`** or run cells sequentially.

## 1. Verify Dual Tesla T4 GPU & PyTorch CUDA Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({round(props.total_memory / (1024**3), 2)} GB VRAM)")

## 2. Sync Canonical AetherOS Worker Runtime

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/crazyBuddhaa/AetherOS.git"
WORK_DIR = Path("/kaggle/working/AetherOS")

if (Path.cwd() / "worker-runtime" / "aether_worker.py").is_file():
    WORK_DIR = Path.cwd()
elif WORK_DIR.exists():
    subprocess.check_call(["git", "-C", str(WORK_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORK_DIR)])

if str(WORK_DIR / "worker-runtime") not in sys.path:
    sys.path.insert(0, str(WORK_DIR / "worker-runtime"))

!pip install -q -r {WORK_DIR}/worker-runtime/requirements.txt
print(f"Canonical runtime ready at {WORK_DIR / 'worker-runtime' / 'aether_worker.py'}")

## 3. Configure Environment & Load Kaggle Secrets

In [ ]:
import os

os.environ.setdefault("AETHER_PLATFORM_URL", "https://aetheros-agent-platform.trileaf.workers.dev")
os.environ.setdefault("CONTRIBUTOR_NAME", "Kaggle Community Contributor")
os.environ.setdefault("AETHER_POLL_INTERVAL", "15")

# Optionally load secrets from Kaggle Add-ons -> Secrets without printing secret values
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
    loaded = []
    for var in ("AETHER_PLATFORM_URL", "AETHER_WORKER_ID", "AETHER_WORKER_KEY", "AETHER_REGISTRATION_TOKEN"):
        try:
            val = secrets_client.get_secret(var)
            if val:
                os.environ[var] = str(val).strip()
                loaded.append(var)
        except Exception:
            pass
    print(f"Kaggle Secrets checked. Loaded keys: {loaded if loaded else 'none (using auto-registration)'}")
except ImportError:
    print("Running outside Kaggle Secrets runtime; using process environment variables.")

print(f"Target Control Plane: {os.environ['AETHER_PLATFORM_URL']}")

## 4. Verify Gateway Connectivity & Launch Authenticated Polling Worker

Executes `aether_worker.run_worker()`: verifies `/api/health`, registers or authenticates via `Authorization: Bearer <key>` + replay nonces, polls `POST /api/workers/tasks/claim`, runs local GPU inference via PyTorch/Transformers, uploads task artifacts to R2, and reports graceful offline shutdown on interrupt.

In [ ]:
import os
import aether_worker

probe = aether_worker.WorkerApiClient(os.environ["AETHER_PLATFORM_URL"])
health = probe.check_health()
print(f"Health verified: status={health.get('status')} | D1={health.get('d1_connected')} | R2={health.get('r2_connected')}")

try:
    aether_worker.run_worker()
except KeyboardInterrupt:
    print("Worker interrupted and shut down cleanly.")